# 🏦 Intelli-Credit — AI Credit Decisioning Engine

## Demo: Sunrise Textile Mills (FY2024)

> **Hackathon Presentation Notebook** — Vivriti Capital AI/ML Hackathon  
> This notebook runs the complete end-to-end pipeline across all 3 persons' modules and generates a professional Credit Appraisal Memorandum (CAM).

### 🔬 11 Innovations Demonstrated
| # | Innovation | Module |
|---|-----------|--------|
| 1 | Beneish M-Score / Altman Z / Piotroski F | Person 1 — Forensics |
| 2 | XGBoost + RF + LightGBM Ensemble PD | Person 1 — ML Scoring |
| 3 | LSTM Trajectory Early Warning | Person 1 — Temporal Model |
| 4 | Promoter Network Graph + Contagion Risk | Person 2 — Network |
| 5 | Sentinel-2 Satellite Activity Scoring | Person 2 — Satellite |
| 6 | GST Filing Intelligence | Person 2 — GST |
| 7 | Monte Carlo Stress Testing (1000 paths) | Person 2 — Stress Test |
| 8 | Corporate DNA Matching (6 archetypes) | Person 2 — DNA |
| 9 | LangGraph Research Agent | Person 3 — Research |
| 10 | Adversarial Bull vs Bear Debate | Person 3 — Agents |
| 11 | Auto-generated CAM DOCX (20 pages) | Person 3 — CAM |

---

In [ ]:
# ── Cell 1: Setup & Imports ──────────────────────────────────────────────────
import os, sys, time, warnings
warnings.filterwarnings("ignore")

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML, Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ Python: {sys.version.split()[0]}")
print(f"✅ Ready to run pipeline")

---
## 🏭 Company Profile — Sunrise Textile Mills

| Parameter | Value |
|-----------|-------|
| **Sector** | Textiles — Cotton Yarn & Fabrics |
| **Revenue** | ₹850 Cr (FY2024) |
| **EBITDA Margin** | 15% |
| **DSCR** | 1.85x |
| **D/E Ratio** | 1.6x |
| **Promoter Holding** | 62% |
| **Location** | Pune, Maharashtra |
| **CIN** | U17100MH2010PLC123456 |

---
## 🔬 Innovation 1 — Financial Forensics (Person 1)
**Beneish M-Score** (earnings manipulation), **Altman Z-Score** (bankruptcy risk), **Piotroski F-Score** (financial strength)

In [ ]:
# ── Innovation 1: Financial Forensics (Person 1) ────────────────────────────
from pipeline.main_pipeline import SUNRISE_DEMO_FINANCIALS, run_layer1_forensics

company_data = SUNRISE_DEMO_FINANCIALS.copy()
forensics = run_layer1_forensics(company_data)

# Display results
m_score = forensics["beneish_m_score"]
z_score = forensics["altman_z_score"]
f_score = forensics["piotroski_f_score"]

m_flag = "🟢 CLEAN" if m_score < -2.22 else "🔴 SUSPICIOUS"
z_flag = "🟢 SAFE" if z_score > 2.99 else ("🟡 GREY ZONE" if z_score > 1.81 else "🔴 DISTRESS")
f_flag = "🟢 STRONG" if f_score >= 7 else ("🟡 MODERATE" if f_score >= 4 else "🔴 WEAK")

display(Markdown(f"""
### Forensic Scores

| Model | Score | Assessment | Threshold |
|-------|-------|------------|-----------|
| **Beneish M-Score** | {m_score:.2f} | {m_flag} | > -2.22 = manipulation |
| **Altman Z-Score** | {z_score:.2f} | {z_flag} | > 2.99 = safe |
| **Piotroski F-Score** | {f_score}/9 | {f_flag} | ≥ 7 = strong |
| **Auditor Distress** | {forensics['auditor_distress_score']}/5 | 🟢 LOW | ≥ 3 = concern |
| **Going Concern** | {"Yes" if forensics['going_concern_flag'] else "No"} | 🟢 CLEAN | — |

> **Verdict**: No earnings manipulation detected. Company is in the grey zone for bankruptcy risk but shows moderate financial strength.
"""))

---
## 🤖 Innovation 2 — ML Credit Scoring (Person 1)
**3-model ensemble**: XGBoost + Random Forest + LightGBM → Probability of Default

In [ ]:
# ── Innovation 2: ML Credit Scoring (Person 1) ──────────────────────────────
from pipeline.main_pipeline import run_layer2_ml_scoring

ml_scores = run_layer2_ml_scoring(company_data)
company_data["ensemble_pd"] = ml_scores["ensemble_pd"]
company_data["lending_decision"] = ml_scores["lending_decision"]

pd_pct = ml_scores["ensemble_pd"] * 100
decision = ml_scores["lending_decision"]
dec_emoji = "🟢" if decision == "APPROVE" else ("🟡" if "CONDITIONAL" in decision else "🔴")

display(Markdown(f"""
### ML Ensemble Credit Score

| Model | PD Score | Weight |
|-------|----------|--------|
| **XGBoost** | {ml_scores['pd_xgb']:.2%} | 40% |
| **Random Forest** | {ml_scores['pd_rf']:.2%} | 30% |
| **LightGBM** | {ml_scores['pd_lgb']:.2%} | 30% |
| **Ensemble** | **{ml_scores['ensemble_pd']:.2%}** | Weighted |

| Metric | Value |
|--------|-------|
| {dec_emoji} **Lending Decision** | **{decision}** |
| 💰 **Credit Limit** | ₹{ml_scores['credit_limit_cr']:.1f} Cr |
| 📈 **Risk Premium** | {ml_scores['risk_premium']:.1f}% |
| ⚠️ **Model Disagreement** | {"Yes" if ml_scores['model_disagreement_flag'] else "No"} |
| 📊 **Source** | {ml_scores['source'].replace('_', ' ').title()} |
"""))

---
## 📈 Innovation 3 — LSTM Trajectory Early Warning (Person 1)
**5-year DSCR trajectory** → Months to danger zone prediction

In [ ]:
# ── Innovation 3: LSTM Trajectory Early Warning (Person 1) ──────────────────
from pipeline.main_pipeline import run_layer3_trajectory

trajectory = run_layer3_trajectory("Sunrise Textile Mills")

level = trajectory["warning_level"]
level_emoji = {"GREEN": "🟢", "YELLOW": "🟡", "ORANGE": "🟠", "RED": "🔴"}.get(level, "⚪")
months = trajectory["estimated_months_to_distress"]

display(Markdown(f"""
### Trajectory Early Warning

| Metric | Value |
|--------|-------|
| {level_emoji} **Warning Level** | **{level}** |
| 🎯 **Risk Score** | {trajectory['trajectory_risk_score']:.4f} |
| ⏱️ **Months to Distress** | {"∞ (improving)" if months > 100 else f"{months:.0f} months"} |
| 📉 **DSCR Velocity** | {trajectory['dscr_velocity']:+.4f}/yr |
| 📊 **Current DSCR** | {trajectory['current_dscr']:.3f} |
"""))

# Plot DSCR trend
dscr_trend = trajectory["dscr_trend"]
years = list(range(2020, 2020 + len(dscr_trend)))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(years, dscr_trend, "o-", color="#1565C0", linewidth=2.5, markersize=8, label="DSCR")
ax.axhline(y=1.0, color="#C62828", linestyle="--", linewidth=1.5, alpha=0.7, label="Danger Zone (1.0)")
ax.axhline(y=1.5, color="#E68A00", linestyle="--", linewidth=1, alpha=0.5, label="Covenant (1.5)")
ax.fill_between(years, 0, 1.0, color="#C62828", alpha=0.08)
ax.set_xlabel("Fiscal Year", fontsize=11)
ax.set_ylabel("DSCR", fontsize=11)
ax.set_title("DSCR Trajectory — Sunrise Textile Mills", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🕸️ Innovation 4 — Promoter Network Graph (Person 2)
**MCA21 Director Linkages** → Bipartite Graph → Contagion Risk Score

In [ ]:
# ── Innovation 4: Promoter Network Graph (Person 2) ─────────────────────────
from modules.person2_alt_data.network_graph import run_network_analysis

t0 = time.time()
network = run_network_analysis("U17100MH2010PLC123456", save_visualization=True)
t_net = time.time() - t0

# Update company_data
company_data["contagion_risk_score"] = network.get("contagion_risk_score", 0)
company_data["network_npa_ratio"] = network.get("npa_ratio", 0)
company_data["promoter_total_companies"] = network.get("promoter_total_companies", 0)
company_data["promoter_npa_companies"] = network.get("promoter_npa_companies", 0)

risk = network.get("risk_level", "N/A")
risk_emoji = "🟢" if risk == "LOW" else ("🟡" if risk == "MEDIUM" else "🔴")

display(Markdown(f"""
### Network Contagion Analysis ({t_net:.1f}s)

| Metric | Value |
|--------|-------|
| {risk_emoji} **Contagion Risk Score** | **{network.get('contagion_risk_score', 0):.4f}** |
| 🏢 **Promoter Companies** | {network.get('promoter_total_companies', 0)} |
| 👤 **Directors in Network** | {network.get('promoter_directors', 0)} |
| ⚠️ **NPA Companies** | {network.get('promoter_npa_companies', 0)} |
| 📊 **NPA Ratio** | {network.get('npa_ratio', 0):.2%} |
| 🔗 **Risk Level** | **{risk}** |
"""))

# Show visualization link
viz_path = network.get("visualization_path")
if viz_path and os.path.exists(viz_path):
    print(f"📊 Interactive network graph saved: {viz_path}")
    print("   (Open in browser to explore the Plotly interactive visualization)")

---
## 🛰️ Innovation 5 — Satellite Activity Scoring (Person 2)
**Sentinel-2 imagery** → NDVI + Brightness + Temporal Delta → Activity Score (0-100)

In [ ]:
# ── Innovation 5: Satellite Activity Scoring (Person 2) ─────────────────────
from modules.person2_alt_data.satellite_module import get_factory_activity

t0 = time.time()
satellite = get_factory_activity(
    company_name="Sunrise Textile Mills",
    revenue_cr=850.0,
    industry_avg_revenue_cr=680.0,
)
t_sat = time.time() - t0

# Update company_data
company_data["satellite_activity_score"] = satellite.get("activity_score", 0)
company_data["satellite_activity_category"] = satellite.get("classification", "N/A")
company_data["satellite_vs_revenue_flag"] = satellite.get("satellite_vs_revenue_flag", 0)

cls = satellite.get("classification", "N/A")
cls_emoji = {"ACTIVE": "🟢", "MODERATE": "🟡", "LOW": "🟠", "DORMANT": "🔴"}.get(cls, "⚪")

display(Markdown(f"""
### Satellite Activity Analysis ({t_sat:.1f}s)

| Metric | Value |
|--------|-------|
| {cls_emoji} **Classification** | **{cls}** |
| 📊 **Activity Score** | **{satellite.get('activity_score', 0):.1f}/100** |
| 🌿 **NDVI Score** | {satellite.get('ndvi_score', 0):.1f} |
| 💡 **Brightness Score** | {satellite.get('brightness_score', 0):.1f} |
| 📡 **Data Source** | {satellite.get('data_source', 'N/A')} |
| 🏭 **Revenue Mismatch** | {"⚠️ YES" if satellite.get('satellite_vs_revenue_flag') else "✅ No"} |
"""))

---
## 🧾 Innovation 6 — GST Filing Intelligence (Person 2)
**Synthetic GST filings** → Revenue divergence + Filing delays + E-way bill consistency → Fraud Risk

In [ ]:
# ── Innovation 6: GST Filing Intelligence (Person 2) ────────────────────────
from modules.person2_alt_data.gst_intelligence import analyze_gst_data

t0 = time.time()
gst = analyze_gst_data("Sunrise Textile Mills", bank_revenue_cr=850.0)
t_gst = time.time() - t0

# Update company_data
company_data["gst_vs_bank_divergence"] = gst.get("gst_vs_bank_divergence", 0)
company_data["gst_divergence_flag"] = gst.get("gst_divergence_flag", 0)
company_data["ewaybill_volume_consistency"] = gst.get("ewaybill_consistency_ratio", 0.9)

fraud = gst.get("fraud_risk_level", "N/A")
fraud_emoji = "🟢" if fraud == "LOW" else ("🟡" if fraud == "MEDIUM" else "🔴")

display(Markdown(f"""
### GST Intelligence ({t_gst:.1f}s)

| Metric | Value |
|--------|-------|
| {fraud_emoji} **Fraud Risk Level** | **{fraud}** |
| 📊 **GST Health Score** | {gst.get('gst_health_score', 0):.1f}/100 |
| 💰 **Bank Revenue** | ₹{gst.get('bank_revenue_cr', 0):.1f} Cr |
| 🧾 **GST Revenue** | ₹{gst.get('gst_annual_revenue_cr', 0):.1f} Cr |
| ↔️ **Divergence** | {gst.get('gst_vs_bank_divergence', 0):.1%} |
| 🚩 **Divergence Flag** | {"⚠️ YES" if gst.get('gst_divergence_flag') else "✅ No"} |
| 📅 **Avg Filing Delay** | {gst.get('avg_filing_delay_days', 0):.1f} days |
| 🚚 **E-Way Bill Consistency** | {gst.get('ewaybill_consistency_ratio', 0):.2%} |
"""))

---
## 🎲 Innovation 7 — Monte Carlo Stress Testing (Person 2)
**1000 macro simulations** → DSCR distribution → P10/P50/P90 + Default Probability

In [ ]:
# ── Innovation 7: Monte Carlo Stress Testing (Person 2) ─────────────────────
from modules.person2_alt_data.stress_test import run_stress_test

t0 = time.time()
stress_financials = {
    "company_name": "Sunrise Textile Mills",
    "revenue": 850.0,
    "ebitda": 127.5,
    "interest_expense": 53.1,   # EBITDA / ICR = 127.5 / 2.4
    "depreciation": 25.5,       # ~20% of EBITDA
    "tax_rate": 0.25,
    "annual_debt_repayment": 78.0,  # ~15% of total debt
}

stress = run_stress_test(stress_financials, n_simulations=1000, save_chart=True)
t_stress = time.time() - t0

# Named scenarios
named = stress.get("named_scenarios", {})
scenarios = named.get("scenarios", [])

display(Markdown(f"""
### Stress Test Results ({t_stress:.1f}s)

#### DSCR Distribution (1000 Monte Carlo Paths)

| Percentile | DSCR | Assessment |
|------------|------|------------|
| **P10 (Severe)** | {stress['p10_dscr']:.4f} | {"🔴" if stress['p10_dscr'] < 1.0 else "🟡"} |
| **P50 (Base)** | {stress['p50_dscr']:.4f} | {"🟢" if stress['p50_dscr'] > 1.2 else "🟡"} |
| **P90 (Optimistic)** | {stress['p90_dscr']:.4f} | 🟢 |
| **Mean ± Std** | {stress['mean_dscr']:.4f} ± {stress['std_dscr']:.4f} | — |

| Risk Metric | Value |
|-------------|-------|
| 🎯 **Default Probability (3yr)** | **{stress['default_probability_3yr']:.2%}** |
| ⚠️ **Covenant Trigger (P20)** | {stress['covenant_trigger_level']:.4f} |
"""))

# Show named scenarios
if scenarios:
    sc_md = "#### Named Stress Scenarios\n\n| Scenario | DSCR | Verdict |\n|----------|------|---------|\n"
    for sc in scenarios:
        v = sc.get("verdict", "")
        emoji = "✅" if v == "PASS" else ("⚠️" if v == "STRESS" else "❌")
        sc_md += f"| {sc['name']} | {sc['dscr']:.4f} | {emoji} {v} |\n"
    display(Markdown(sc_md))

# Show the stress distribution chart
chart_path = stress.get("chart_path")
if chart_path and os.path.exists(chart_path):
    display(Markdown("#### Stress Distribution Chart"))
    display(Image(filename=chart_path, width=700))

---
## 🧬 Innovation 8 — Corporate DNA Matching (Person 2)
**Cosine similarity** against 6 Indian collapse archetypes: IL&FS, DHFL, Jet Airways, Videocon, Satyam, Kingfisher

In [ ]:
# ── Innovation 8: Corporate DNA Matching (Person 2) ─────────────────────────
from modules.person2_alt_data.dna_matching import run_dna_analysis

t0 = time.time()
dna = run_dna_analysis(company_name="Sunrise Textile Mills")
t_dna = time.time() - t0

risk = dna.get("borrower_risk_profile", "N/A")
risk_emoji = {"LOW": "🟢", "MODERATE": "🟡", "ELEVATED": "🟠", "HIGH": "🔴", "CRITICAL": "🔴"}.get(risk, "⚪")

display(Markdown(f"""
### Corporate DNA Analysis ({t_dna:.1f}s)

| Metric | Value |
|--------|-------|
| {risk_emoji} **Risk Profile** | **{risk}** |
| 🎯 **Closest Archetype** | {dna.get('closest_archetype', 'N/A')} |
| 📊 **Max Similarity** | {dna.get('max_similarity', 0):.4f} |
"""))

# Archetype similarity table
details = dna.get("archetype_details", [])
if details:
    tbl = "#### Similarity to Collapse Archetypes\n\n"
    tbl += "| Archetype | Similarity | Features Matched | Risk |\n"
    tbl += "|-----------|-----------|-----------------|------|\n"
    for ad in details:
        s = ad["similarity_score"]
        emoji = "🔴" if s > 0.75 else ("🟡" if s > 0.50 else "🟢")
        tbl += f"| {ad['archetype']} | {s:.4f} | {ad['matched_features']} | {emoji} |\n"
    display(Markdown(tbl))

# DNA warning
warning = dna.get("warning")
if warning:
    display(Markdown(f"⚠️ **DNA Warning:**\n```\n{warning}\n```"))

---
## 🔍 Innovation 9 — LangGraph Research Agent (Person 3)
**Tavily Web Search → Claude Intelligence Extraction → Structured Research Summary**  
*(Falls back to synthetic data when API keys unavailable)*

In [ ]:
# ── Innovation 9: Research Agent (Person 3) ──────────────────────────────────
from modules.person3_llm_cam.research_agent import run_research

t0 = time.time()
research = run_research("Sunrise Textile Mills", sector="Textiles", promoter_name="Promoter Group")
t_res = time.time() - t0

outlook = research.get("industry_outlook", "N/A")
out_emoji = {"POSITIVE": "🟢", "NEUTRAL": "🟡", "NEGATIVE": "🔴"}.get(outlook, "⚪")
fallback = "Synthetic Fallback" if research.get("used_fallback") else "Live API"

display(Markdown(f"""
### Research Intelligence ({t_res:.1f}s)

| Metric | Value |
|--------|-------|
| {out_emoji} **Industry Outlook** | **{outlook}** |
| 📊 **Sentiment Score** | {research.get('research_sentiment_score', 0):.2f} |
| 🔗 **Data Source** | {fallback} |
"""))

# Key positives
positives = research.get("key_positives_found", [])
if positives:
    md = "#### ✅ Key Positives\n"
    for p in positives:
        md += f"- {p}\n"
    display(Markdown(md))

# Key risks
risks = research.get("key_risks_found", [])
if risks:
    md = "#### ⚠️ Key Risks\n"
    for r in risks:
        md += f"- {r}\n"
    display(Markdown(md))

# News summary
news = research.get("company_news_summary", "")
if news:
    display(Markdown(f"#### 📰 News Summary\n> {news[:500]}"))

---
## ⚔️ Innovation 10 — Adversarial Bull vs Bear Debate (Person 3)
**Approval Agent** (Bull Case) vs **Dissent Agent** (Bear Case) → **Coordinator** synthesizes final recommendation

In [ ]:
# ── Innovation 10: Adversarial Bull vs Bear + CEO Interview (Person 3) ──────
from modules.person3_llm_cam.approval_agent import write_bull_case
from modules.person3_llm_cam.dissent_agent import write_bear_case, synthesize_cam_recommendation
from modules.person3_llm_cam.ceo_interview import run_ceo_interview_analysis

# CEO Interview (fallback — no audio file)
t0 = time.time()
ceo_interview = run_ceo_interview_analysis(company_data=company_data)
t_ceo = time.time() - t0

# Bull case (approval agent)
t0 = time.time()
bull_case = write_bull_case(company_data, research)
t_bull = time.time() - t0

# Bear case (dissent agent)
t0 = time.time()
bear_case = write_bear_case(company_data, bull_case, research)
t_bear = time.time() - t0

# Coordinator synthesizes recommendation
t0 = time.time()
scores = {
    "ensemble_pd": ml_scores["ensemble_pd"],
    "dscr": company_data.get("dscr", 1.85),
    "lending_decision": ml_scores["lending_decision"],
    "risk_premium": ml_scores["risk_premium"],
    "revenue": company_data.get("revenue", 850.0),
}
recommendation = synthesize_cam_recommendation(bull_case, bear_case, scores)
t_rec = time.time() - t0

decision = recommendation.get("lending_decision", "N/A")
dec_emoji = "🟢" if decision == "APPROVE" else ("🟡" if "CONDITIONAL" in decision else "🔴")

display(Markdown(f"""
### Adversarial Debate Results

| Agent | Time |
|-------|------|
| 🎤 CEO Interview | {t_ceo:.1f}s |
| 🐂 Bull Case (Approval) | {t_bull:.1f}s |
| 🐻 Bear Case (Dissent) | {t_bear:.1f}s |
| ⚖️ Coordinator Synthesis | {t_rec:.1f}s |

---

### {dec_emoji} Final Recommendation: **{decision}**

| Metric | Value |
|--------|-------|
| 💰 **Credit Limit** | ₹{recommendation.get('recommended_limit_cr', 'N/A')} Cr |
| 📈 **Interest Rate** | {recommendation.get('recommended_rate_pct', 'N/A')}% |
"""))

# Show conditions
conditions = recommendation.get("key_conditions", [])
if conditions:
    md = "#### 📋 Key Conditions\n"
    for c in conditions:
        md += f"1. {c}\n"
    display(Markdown(md))

# Bull vs Bear side by side
display(Markdown("---\n### 🐂 Bull Case (Approval Agent)"))
display(Markdown(bull_case[:1500] + ("\n\n*[truncated for display]*" if len(bull_case) > 1500 else "")))

display(Markdown("---\n### 🐻 Bear Case (Dissent Agent)"))
display(Markdown(bear_case[:1500] + ("\n\n*[truncated for display]*" if len(bear_case) > 1500 else "")))

# Final rationale
rationale = recommendation.get("final_rationale", "")
if rationale:
    display(Markdown(f"---\n### ⚖️ Coordinator's Rationale\n> {rationale}"))

---
## 📄 Innovation 11 — Auto-Generated CAM Document (Person 3)
**Professional 20-page Credit Appraisal Memorandum** — Assembles all 10 innovations into a formatted DOCX

In [ ]:
# ── Innovation 11: CAM Document Generation (Person 3) ───────────────────────
from modules.person3_llm_cam.cam_generator import generate_cam

t0 = time.time()

# Assemble all data for CAM generator
cam_data = {
    "company_name": "Sunrise Textile Mills",
    "fiscal_year": 2024,
    "sector": "Textiles",
    "financial_data": company_data,
    "forensics": forensics,
    "ml_scores": ml_scores,
    "trajectory": trajectory,
    "network": network,
    "satellite": satellite,
    "gst": gst,
    "stress_test": {
        "dscr_p10": stress.get("p10_dscr"),
        "dscr_p50": stress.get("p50_dscr"),
        "dscr_p90": stress.get("p90_dscr"),
        "covenant_breach_probability": stress.get("default_probability_3yr"),
        "named_scenarios": [
            {
                "name": sc.get("name", ""),
                "dscr_impact": sc.get("dscr", 0),
                "pd_impact": max(0, 1.0 - sc.get("dscr", 1.0)),
            }
            for sc in stress.get("named_scenarios", {}).get("scenarios", [])
        ],
    },
    "dna_match": {
        "closest_default_archetype": dna.get("closest_archetype", "N/A"),
        "max_archetype_similarity": dna.get("max_similarity", 0),
    },
    "research": research,
    "ceo_interview": ceo_interview,
    "bull_case": bull_case,
    "bear_case": bear_case,
    "recommendation": recommendation,
}

output_dir = os.path.join(PROJECT_ROOT, "data", "processed")
cam_path = generate_cam(cam_data, output_dir=output_dir)
t_cam = time.time() - t0

if cam_path and os.path.exists(cam_path):
    file_size_kb = os.path.getsize(cam_path) / 1024
    display(Markdown(f"""
### ✅ CAM Generated Successfully ({t_cam:.1f}s)

| Property | Value |
|----------|-------|
| 📄 **File** | `{os.path.basename(cam_path)}` |
| 📁 **Path** | `{cam_path}` |
| 📏 **Size** | {file_size_kb:.1f} KB |
| 📊 **Sections** | 11 (Cover → Recommendation) |
"""))
else:
    display(Markdown("❌ CAM generation failed. Check that `python-docx` is installed."))

---
## 🏁 Pipeline Summary & Download

In [ ]:
# ── Pipeline Summary & Download Button ───────────────────────────────────────
import base64

display(Markdown(f"""
# 🏁 Intelli-Credit Pipeline — Complete

## Sunrise Textile Mills (FY2024)

### Decision Summary

| Layer | Module | Key Result |
|-------|--------|------------|
| 1 | Forensics | M-Score: {forensics['beneish_m_score']:.2f} (CLEAN) |
| 2 | ML Scoring | PD: {ml_scores['ensemble_pd']:.2%} → {ml_scores['lending_decision']} |
| 3 | Trajectory | {trajectory['warning_level']} ({trajectory['trajectory_risk_score']:.4f}) |
| 4 | Network | Contagion: {network.get('contagion_risk_score', 0):.4f} ({network.get('risk_level', 'N/A')}) |
| 5 | Satellite | Score: {satellite.get('activity_score', 0):.1f}/100 ({satellite.get('classification', 'N/A')}) |
| 6 | GST Intel | Health: {gst.get('gst_health_score', 0):.1f}/100 (Fraud: {gst.get('fraud_risk_level', 'N/A')}) |
| 7 | Stress Test | P50 DSCR: {stress.get('p50_dscr', 0):.4f}, Default Prob: {stress.get('default_probability_3yr', 0):.2%} |
| 8 | DNA Match | Closest: {dna.get('closest_archetype', 'N/A')} ({dna.get('max_similarity', 0):.4f}) |
| 9 | Research | Outlook: {research.get('industry_outlook', 'N/A')} |
| 10 | Bull vs Bear | {decision} (₹{recommendation.get('recommended_limit_cr', 'N/A')} Cr @ {recommendation.get('recommended_rate_pct', 'N/A')}%) |
| 11 | CAM DOCX | ✅ Generated |

---
"""))

# Download CAM button
if cam_path and os.path.exists(cam_path):
    with open(cam_path, "rb") as f:
        cam_bytes = f.read()
    b64 = base64.b64encode(cam_bytes).decode()
    filename = os.path.basename(cam_path)

    download_html = f"""
    <div style="text-align: center; padding: 20px;">
        <a href="data:application/vnd.openxmlformats-officedocument.wordprocessingml.document;base64,{b64}"
           download="{filename}"
           style="
               background: linear-gradient(135deg, #0A1F3C, #14396B);
               color: white;
               padding: 16px 40px;
               font-size: 18px;
               font-weight: bold;
               text-decoration: none;
               border-radius: 8px;
               display: inline-block;
               box-shadow: 0 4px 15px rgba(10, 31, 60, 0.4);
               transition: transform 0.2s;
           "
           onmouseover="this.style.transform='scale(1.05)'"
           onmouseout="this.style.transform='scale(1.0)'"
        >
            📥 Download CAM — {filename}
        </a>
        <p style="color: #757575; margin-top: 12px; font-size: 13px;">
            Credit Appraisal Memorandum • {os.path.getsize(cam_path)/1024:.0f} KB • DOCX Format
        </p>
    </div>
    """
    display(HTML(download_html))
else:
    print("⚠️ CAM file not found — install python-docx and re-run Innovation 11")

display(Markdown("""
---
> **Built with** 🤖 3 ML models + 🛰️ Satellite imagery + 🕸️ Network graphs + 🧬 DNA matching +
> 🎲 Monte Carlo + 🤖 LLM agents + 📄 Auto-generated CAM
>
> **Intelli-Credit** — Vivriti Capital AI/ML Hackathon 2026
"""))